# Full HIGGS Benchmark on Colab or Kaggle

**Author:** CosmicMagnetar | **Project:** Higgs Event Lab

This notebook runs the full Higgs benchmark on a large sample using the official UCI data link. It fetches data automatically, trains logistic regression, XGBoost, and a wide compact MLP on one frozen test split, and saves model weights plus predictions for your report.


## Step 1: Prepare the environment

Run this cell first before any other cell. On a fresh Colab or Kaggle runtime the notebook clones this project repository once, installs any missing packages such as xgboost and torch, and adds the project source folder to the Python path so every later cell can import the shared data and model code without errors.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/CosmicMagnetar/higgs-event-lab.git"

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
if not (root / "src" / "higgs_event_lab").exists():
    print("Cloning repository into", root)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)

print("Project root:", root)

for package in ("numpy", "pandas", "scikit-learn", "xgboost", "torch", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        print("Installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)

if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

print("Environment ready.")


## Step 2: Download the full HIGGS dataset

This cell fetches the original UCI HIGGS archive and extracts the compressed CSV. It skips the download when the file already exists. The archive is about 2.6 GB, so the first run takes time.


In [ ]:
import os
from pathlib import Path
import shutil
import sys
import urllib.request
import zipfile

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
raw_path.parent.mkdir(parents=True, exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"

if not raw_path.exists():
    print("Downloading full HIGGS from UCI...")
    zip_path = raw_path.parent / "higgs_download.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        gz_name = [name for name in archive.namelist() if name.endswith(".csv.gz")][0]
        with archive.open(gz_name) as source, open(raw_path, "wb") as target:
            shutil.copyfileobj(source, target)
    zip_path.unlink()
print("Raw data ready:", raw_path)


## Step 3: Prepare a one million event subset

This cell builds a processed sample for full-scale training using the shared extraction helper. Adjust the row count if memory is limited.


In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from higgs_event_lab.data import extract_sample

raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
out_path = root / "data" / "processed" / "higgs_sample_1m.csv.gz"

if not out_path.exists():
    extract_sample(source_path=raw_path, output_path=out_path, n_rows=1000000, compare_later_chunk=False)
    print("Processed sample ready:", out_path)


## Research and authorship

Saumya Mishra owns the physics background, dataset guide, and QA plan that explain why the 21 low-level and 7 high-level features are the right problem here. CosmicMagnetar tuned the models and prepared this Colab walkthrough so the numbers can be reproduced without claiming physics the benchmark never measures. Use this note in your report and viva so the reader understands who wrote what.


## Step 4: Train the tuned full benchmark

This cell launches logistic regression, XGBoost, and a wider compact MLP on the processed sample. It keeps the test split frozen across models and writes results, predictions, and saved model weights into the outputs folder.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
script = root / "scripts" / "train_full_colab.py"

os.environ["HIGGS_SAMPLE_PATH"] = str(root / "data" / "processed" / "higgs_sample_1m.csv.gz")
result = subprocess.run([sys.executable, str(script)], cwd=root, check=False)
print("Tuned training exit code:", result.returncode)


## Step 5: Show the results

This cell reads the saved training summary from outputs/full_colab_results.json so you can copy the ROC-AUC, PR-AUC, Brier score, accuracy, and training time straight into your report. It also lists every artifact path, including the saved model weights.


In [ ]:
import json
from pathlib import Path

root = Path(os.environ.get("HIGGS_REPO_ROOT", Path.cwd() / "higgs-event-lab"))
summary_path = root / "outputs" / "full_colab_results.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Sample rows:", summary["rows"])
    print("Split:", summary["split"])
    print()
    for record in summary["runs"]:
        print(f"{record['model']:<28} ROC-AUC {record['test_roc_auc']:.4f}  PR-AUC {record['test_pr_auc']:.4f}  Brier {record['brier_score']:.4f}  accuracy {record['test_accuracy']:.4f}  seconds {record['training_seconds']:.1f}")
    print()
    print("Artifacts:")
    for artifact in summary["artifacts"]:
        print(" ", artifact)
else:
    print("No summary found yet. Run Step 4 first.")
